In [ ]:
# Core libs for file handling, arrays, dataframes, and plotting
import os  # filesystem paths and directory utilities
import numpy as np  # vectorized numerical ops
import pandas as pd  # tabular data handling
import seaborn as sns  # statistical plotting
sns.set_style('darkgrid')  # consistent dark grid background for plots
import matplotlib.pyplot as plt  # plotting primitives
from sklearn.model_selection import train_test_split  # dataset splitting helper
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)  # evaluation metrics

from PIL import Image  # image I/O and basic manipulation

# Deep learning stack (TensorFlow/Keras) and computer vision helpers
import tensorflow as tf  # core TensorFlow engine
from tensorflow import keras  # Keras high-level API
from tensorflow.keras.models import Sequential  # linear stack of layers
from tensorflow.keras.optimizers import Adam, Adamax  # optimizers
from tensorflow.keras.metrics import categorical_crossentropy  # loss metric alias
from tensorflow.keras.preprocessing.image import ImageDataGenerator  # augmentation pipeline
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Activation, Dropout, BatchNormalization  # layer building blocks
from tensorflow.keras import regularizers  # weight regularization
from pathlib import Path
import shutil
from sklearn.model_selection import train_test_split

import cv2  # OpenCV for image processing
import warnings  # control warnings
from tqdm import tqdm  # progress bars

# Silence noisy warnings to keep notebook output clean
warnings.filterwarnings("ignore")

print('modules loaded')  # quick sanity check that imports succeeded

## Dataset locations
Define dataset root paths for train/val/test splits.

In [ ]:
# Define base dataset path for the two datasets
dataset1_path = "dataset1"  # root folder holding train/val/test subfolders for dataset 1
dataset2_path = "dataset2"  # root folder holding train/val/test subfolders for dataset 2

## Re-split dataset to 70/20/10
Regenerate balanced train/val/test folders under `dataset_70_20_10`.

In [ ]:
from pathlib import Path
import shutil
from sklearn.model_selection import train_test_split

def ensure_dir(path: Path):
    if not path.exists():
        path.mkdir(parents=True, exist_ok=True)

def resplit_dataset(source_root: Path, target_root: Path):
    # start clean
    if target_root.exists():
        shutil.rmtree(target_root)

    # collect all images per class across current splits
    class_dirs = sorted(p.name for p in (source_root / "train").iterdir() if p.is_dir())
    all_images = {cls: [] for cls in class_dirs}

    valid_exts = [".jpg", ".jpeg", ".png"]

    for split in ["train", "val", "test"]:
        for cls in class_dirs:
            split_cls_dir = source_root / split / cls
            if split_cls_dir.exists():
                all_images[cls].extend([
                    p for p in split_cls_dir.iterdir()
                    if p.is_file() and p.suffix.lower() in valid_exts
                ])

    # DEBUG: show counts before splitting
    print("\n📊 Image counts before splitting:")
    for cls, imgs in all_images.items():
        print(f"  {cls}: {len(imgs)} images")

    # split 70/20/10 using train_test_split
    new_counts = {"train": {}, "val": {}, "test": {}}

    for cls, imgs in all_images.items():
        # 🚫 Skip empty classes
        if len(imgs) == 0:
            print(f"⚠️ Skipping class '{cls}' (no images found)")
            continue

        # ⚠️ Handle very small datasets safely
        if len(imgs) < 3:
            print(f"⚠️ Skipping class '{cls}' (too few images: {len(imgs)})")
            continue

        try:
            train_imgs, tmp_imgs = train_test_split(
                imgs, test_size=0.3, random_state=42, shuffle=True
            )

            val_imgs, test_imgs = train_test_split(
                tmp_imgs, test_size=2/3, random_state=42, shuffle=True
            )

        except ValueError as e:
            print(f"⚠️ Skipping class '{cls}' due to split error: {e}")
            continue

        split_map = {
            "train": train_imgs,
            "val": val_imgs,
            "test": test_imgs
        }

        for split, split_imgs in split_map.items():
            new_counts[split][cls] = len(split_imgs)

            for img_path in split_imgs:
                target_path = target_root / split / cls / img_path.name
                ensure_dir(target_path.parent)
                shutil.copy(img_path, target_path)

    print("\n✅ Re-split complete ->", target_root)
    print("📊 Counts per split:")

    for split, cls_counts in new_counts.items():
        print(f"  {split}:")
        for cls, count in cls_counts.items():
            print(f"    {cls}: {count} images")

    return str(target_root)


# Paths
source_root1 = Path(dataset1_path)
target_root1 = Path("dataset1_70_20_10")

source_root2 = Path(dataset2_path)
target_root2 = Path("dataset2_70_20_10")

# Run for both datasets
dataset1_path = resplit_dataset(source_root1, target_root1)
dataset2_path = resplit_dataset(source_root2, target_root2)

## Class list and counts
List class folders, tally images per split, and sanity-check balance.

In [ ]:
# Inspect available class folders under the training split for both datasets
print("\n Available classes in training splits:")
for cls in sorted((Path(dataset1_path) / "train").iterdir()):
    if cls.is_dir():
        print(f" for dataset 1")
        print(f"  {cls.name}")

for cls in sorted((Path(dataset2_path) / "train").iterdir()):
    if cls.is_dir():
        print(f" for dataset 2")
        print(f"  {cls.name}")

In [ ]:
# Helper to count images per class inside a split folders for both datasets for training, test and evaluation
def count_images_per_class(split_path: Path):
    class_counts = {}
    for cls_dir in split_path.iterdir():
        if cls_dir.is_dir():
            count = sum(1 for p in cls_dir.iterdir() if p.is_file())
            class_counts[cls_dir.name] = count
    return class_counts
        
# Count images per class for training, validation and test splits for both datasets
print("\n Image counts per class in training split:")
train_counts1 = count_images_per_class(Path(dataset1_path) / "train")
train_counts2 = count_images_per_class(Path(dataset2_path) / "train")
print(f" for dataset 1: {train_counts1}")
print(f" for dataset 2: {train_counts2}")
print("\n Image counts per class in validation split:")
val_counts1 = count_images_per_class(Path(dataset1_path) / "val")
val_counts2 = count_images_per_class(Path(dataset2_path) / "val")
print(f" for dataset 1: {val_counts1}")
print(f" for dataset 2: {val_counts2}")
print("\n Image counts per class in test split:")
test_counts1 = count_images_per_class(Path(dataset1_path) / "test")
test_counts2 = count_images_per_class(Path(dataset2_path) / "test")
print(f" for dataset 1: {test_counts1}")
print(f" for dataset 2: {test_counts2}")

## Visualize training distribution
Bar chart of class counts in the training split.

In [ ]:
# Visualize class distribution in the training split for both datasets using bar plots
plt.figure(figsize=(16,6))

plt.subplot(1,3,1)
plt.bar(train_counts1.keys(), train_counts1.values())
plt.xlabel("Class") # x-axis label
plt.ylabel("Count") # y-axis label
plt.title("Dataset 1") # title

plt.subplot(1,3,2)
plt.bar(train_counts2.keys(), train_counts2.values())
plt.xlabel("Class") # x-axis label
plt.ylabel("Count") # y-axis label
plt.title("Dataset 2") # title

plt.show()

In [ ]:
# Build a dataframe listing every image with its split and label for both datasets to facilitate EDA and later loading
records = []  # accumulator rows

for split in ['train','val','test']:
    for dataset_path in [dataset1_path, dataset2_path]:
        dataset_name = Path(dataset_path).name
        split_path = Path(dataset_path) / split
        
        for cls_dir in split_path.iterdir():
            if cls_dir.is_dir():
                cls_name = cls_dir.name
                for img_path in cls_dir.iterdir():
                    if img_path.is_file():
                        records.append({
                            "dataset": dataset_name,
                            "split": split,
                            "class": cls_name,
                            "image_path": str(img_path)
                        })

eda_df = pd.DataFrame(records)  # assemble rows into dataframe

print("Total images:", len(eda_df))

## Build EDA dataframe
Enumerate all files with split/label metadata for downstream analysis.

In [ ]:
# Sample image dimensions to understand typical sizes
sizes = []  # holds (width, height) tuples

sample_paths = eda_df['image_path'].sample(200)  # random subset of images

for path in sample_paths:
    try:
        with Image.open(path) as img:
            sizes.append(img.size)  # (width, height)
    except Exception as e:
        print(f"⚠️ Could not open {path}: {e}")
        
# Analyze size distribution
size_df = pd.DataFrame(sizes, columns=['width', 'height'])
print("\nImage size statistics:")
print(size_df.describe())

## Image dimension sampling
Sample image sizes to choose a sensible target resize.

In [ ]:
# Inspect pixel value range on a sample image
# load first image into array to check pixel value range (should be 0-255 for typical RGB images)
sample_image_path = eda_df['image_path'].iloc[0]
try:
    with Image.open(sample_image_path) as img:
        img_array = np.array(img)
        print(f"\nSample image shape: {img_array.shape}")
        print(f"Pixel value range: {img_array.min()} to {img_array.max()}")
except Exception as e:
    print(f"⚠️ Could not open sample image {sample_image_path}: {e}")
    
print("Min pixel value:",   img_array.min())
print("Max pixel value:",   img_array.max())

## Pixel range sanity check
Confirm images are in expected 0–255 range.

In [ ]:
# Show a few example images per class to spot visual patterns
samples_per_class = 3  # images per label to display

for label in eda_df['class'].unique():
    sample_paths = eda_df[(eda_df['class'] == label) & (eda_df['split'] == 'train')]['image_path'].sample(samples_per_class, random_state=42)
    
    plt.figure(figsize=(12,4))
    for i, path in enumerate(sample_paths):
        try:
            with Image.open(path) as img:
                plt.subplot(1, samples_per_class, i+1)
                plt.imshow(img)
                plt.title(f"{label}")
                plt.axis('off')
        except Exception as e:
            print(f"⚠️ Could not open {path}: {e}")
    plt.suptitle(f"Sample images for class: {label}", fontsize=16)
    plt.show()

## Visual samples per class
Show a few images from each class to spot issues or patterns.

In [ ]:
broken = []  # collect paths that fail to open

for path in eda_df['image_path']:
    try:
        with Image.open(path) as img:
            img.verify()  # verify integrity without loading full image
    except Exception as e:
        broken.append((path, str(e)))
        
print(f"\nTotal unreadable/corrupted images: {len(broken)}")
if broken:
    print("Examples of broken images:")
    for path, error in broken[:5]:  # show first 5 examples
        print(f"  {path}: {error}")

In [ ]:
# Check for unreadable/corrupted images for all datasets and splits to ensure data integrity before training
for dataset_path in [dataset1_path, dataset2_path]:
    for split in ['train','val','test']:
        split_path = Path(dataset_path) / split
        for cls_dir in split_path.iterdir():
            if cls_dir.is_dir():
                for img_path in cls_dir.iterdir():
                    if img_path.is_file():
                        try:
                            with Image.open(img_path) as img:
                                img.verify()  # check if image can be opened and is not corrupted
                        except Exception as e:
                            print(f"⚠️ Unreadable/corrupted image found: {img_path} -> {e}")
print("Data integrity check complete.")


## Corruption check
Attempt to open every file and flag unreadable images.

In [ ]:
# Build a sampled stats dataframe capturing geometry, color, and texture features per image for both datasets to facilitate EDA and later loading
tqdm.pandas()  # enable pandas integration for progress bars

def extract_features(image_path):
    try:
        with Image.open(image_path) as img:
            img = img.convert('RGB')  # ensure RGB format
            img_array = np.array(img)
            
            # Geometry features
            width, height = img.size
            aspect_ratio = width / height
            
            # Color features
            mean_color = img_array.mean(axis=(0,1))  # mean per channel
            std_color = img_array.std(axis=(0,1))    # std per channel
            
            # Texture features (using simple edge detection as a proxy)
            gray = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY)
            edges = cv2.Canny(gray, 100, 200)
            edge_density = edges.sum() / (width * height)  # proportion of edge pixels
            
            return {
                "width": width,
                "height": height,
                "aspect_ratio": aspect_ratio,
                "mean_red": mean_color[0],
                "mean_green": mean_color[1],
                "mean_blue": mean_color[2],
                "std_red": std_color[0],
                "std_green": std_color[1],
                "std_blue": std_color[2],
                "edge_density": edge_density
            }
    except Exception as e:
        print(f"⚠️ Could not process {image_path}: {e}")
        return None
    
stat_df = eda_df['image_path'].progress_apply(extract_features).dropna().apply(pd.Series) # expand dicts into columns
stat_df = pd.concat([eda_df, stat_df], axis=1)  # combine   with original dataframe for easier analysis

print(stat_df.head())
print('\nSample size used:', len(stat_df))

## Image-level feature extraction
Compute geometry, color, and texture stats on a sampled subset for downstream visuals.

**Why Visual 1 & 2 (count plots)**: Bar counts reveal whether splits or labels are imbalanced. They use seaborn `countplot` to tally rows from `eda_df`, showing split and class frequencies. Without these, we could miss data imbalance that skews training/validation and metrics.

In [ ]:
# Visual 1: overall split distribution for both datasets using bar plots to check for class imbalance
# visualisation for dataset 1
plt.figure(figsize=(16,6))
sns.countplot(data=eda_df, x='split', hue='dataset', order=['train','val','test'], palette='Set2')  # bars per split    
plt.title('Images per split')
plt.xlabel('Split')
plt.ylabel('Count')
plt.show()


## Split and class distributions
Count plots for splits and labels to spot imbalance.

**Why Visual 3 & 4 (stacked/grouped bars)**: These compare class counts across train/val/test in one view using the precomputed `counts` table. Stacked shows total volume; grouped highlights per-split differences. Skipping them hides split-specific imbalance that can cause biased evaluation or sampling issues.

In [ ]:
# Compare class counts per split using stacked and grouped bars for both datasets to check for class imbalance and distribution differences
plt.figure(figsize=(16,6))
sns.countplot(data=eda_df, x='split', hue='class', order=['train','val','test'], palette='Set2')  # bars per split    
plt.title('Class distribution per split')
plt.xlabel('Split')
plt.ylabel('Count')
plt.show()

## Split/class comparison
Stacked and grouped bars to compare class counts across splits.

**Why Visual 5 (heatmap)**: Heatmap of the `counts` matrix quickly spotlights over/under-represented class-split combos with color intensity. Without it, we rely on numbers alone and may overlook sparse cells that can cause poor generalization for specific classes in a split.

In [ ]:
# Heatmap to highlight count imbalances across split/label for both datasets
pivot_df = eda_df.pivot_table(index='class', columns='split', aggfunc='size', fill_value=0)  # create pivot table of counts
plt.figure(figsize=(10,6))
sns.heatmap(pivot_df, annot=True, fmt='d', cmap='YlGnBu')  # color-coded counts
plt.title('Visualizing Heatmap of class counts by split')
plt.xlabel('Label')
plt.ylabel('Split')
plt.show()

## Imbalance heatmap
Heatmap of class counts per split to highlight sparse cells.

**Why Visual 6 & 7 (pie charts)**: Pies summarize proportion of labels overall and splits overall, emphasizing share instead of counts. They make imbalance intuitive at a glance. Omitting them removes an easy percentage view; you’d rely on bar heights only, which can hide proportion differences when totals vary.

In [ ]:
# Pie charts summarizing class and split proportions for both dataset1 and dataset2
plt.figure(figsize=(16,6))
plt.subplot(1,2,1)
eda_df['class'].value_counts().plot(kind='pie', autopct='%1.1f%%', colors=sns.color_palette('Set3'))  # class share
plt.ylabel('')
plt.title('Class share (all splits)')

plt.subplot(1,2,2)
eda_df['split'].value_counts().reindex(['train','val','test']).plot(kind='pie', autopct='%1.1f%%', colors=sns.color_palette('Set2'))  # split share
plt.ylabel('')
plt.title('Split share')
plt.show()

## Proportion pies
Pie charts to show label and split share as percentages.

**Why Visual 8 & 9 (width/height histograms)**: Histograms of dimensions show common image sizes and spread, guiding resize strategy. Without them we might choose an input size that crops or stretches many images, hurting model quality.

In [ ]:
# Width and height distributions from sampled stats
plt.figure(figsize=(6,4))
sns.histplot(stat_df['width'], bins=30, color='steelblue', kde=True)  # histogram of widths
plt.title('Width distribution (sample)')
plt.xlabel('Width (px)')
plt.ylabel('Frequency')
plt.show()

plt.figure(figsize=(6,4))
sns.histplot(stat_df['height'], bins=30, color='salmon', kde=True)  # histogram of heights
plt.title('Height distribution (sample)')
plt.xlabel('Height (px)')
plt.ylabel('Frequency')
plt.show()

## Geometry histograms
Width and height distributions from sampled images.

**Why Visual 10 & 11 (hexbin + aspect histogram)**: Hexbin shows joint density of width vs height to spot typical resolutions and outliers; the aspect histogram shows shape tendencies. Skipping them hides whether sizes cluster along certain diagonals or if extreme aspect ratios need special handling.

In [ ]:
# Joint view of width vs height plus aspect ratio distribution
plt.figure(figsize=(6,5))
plt.hexbin(stat_df['width'], stat_df['height'], gridsize=25, cmap='viridis')  # 2D density of sizes
plt.title('Width vs Height density')
plt.xlabel('Width (px)')
plt.ylabel('Height (px)')
cb = plt.colorbar()
cb.set_label('Count')
plt.show()

plt.figure(figsize=(6,4))
sns.histplot(stat_df['aspect'], bins=30, color='mediumseagreen', kde=True)  # aspect ratio distribution
plt.title('Aspect ratio (W/H) distribution')
plt.xlabel('Aspect ratio')
plt.ylabel('Frequency')
plt.show()

## Size density and aspect ratios
Hexbin of width/height and histogram of aspect ratios.

**Why Visual 12–14 (boxplots width/height/aspect by label)**: Boxplots compare geometry across classes to detect if one class has systematically different image sizes or shapes. Without them, class-specific preprocessing needs could be missed, leading to distortions for certain labels.

In [ ]:
# Boxplots to compare geometry across labels
plt.figure(figsize=(7,4))
sns.boxplot(data=stat_df, x='label', y='width', order=sorted(stat_df['label'].unique()), palette='Pastel1')  # width per class
plt.title('Width by label (sample)')
plt.xlabel('Label')
plt.ylabel('Width (px)')
plt.xticks(rotation=20)
plt.show()

plt.figure(figsize=(7,4))
sns.boxplot(data=stat_df, x='label', y='height', order=sorted(stat_df['label'].unique()), palette='Pastel2')  # height per class
plt.title('Height by label (sample)')
plt.xlabel('Label')
plt.ylabel('Height (px)')
plt.xticks(rotation=20)
plt.show()

plt.figure(figsize=(7,4))
sns.boxplot(data=stat_df, x='label', y='aspect', order=sorted(stat_df['label'].unique()), palette='Set2')  # aspect ratio per class
plt.title('Aspect ratio by label (sample)')
plt.xlabel('Label')
plt.ylabel('Aspect ratio (W/H)')
plt.xticks(rotation=20)
plt.show()

## Geometry by label
Boxplots comparing width, height, and aspect ratio across classes.

**Why Visual 15 & 16 (orientation counts, area histogram)**: Orientation counts reveal dominant landscape/portrait mix, informing augmentation strategy; area histogram shows pixel count scale. Skipping them risks choosing crops/resize policies that penalize minority orientations or overwhelm memory with very large images.

In [ ]:
# Orientation mix and overall pixel area distribution
plt.figure(figsize=(5,4))
sns.countplot(data=stat_df, x='orientation', order=['landscape','portrait','square'], palette='coolwarm')  # orientation counts
plt.title('Orientation distribution (sample)')
plt.xlabel('Orientation')
plt.ylabel('Count')
plt.show()

plt.figure(figsize=(6,4))
sns.histplot(stat_df['area'], bins=30, color='mediumpurple', kde=True)  # pixel area histogram
plt.title('Image area distribution (sample)')
plt.xlabel('Area (px^2)')
plt.ylabel('Frequency')
plt.show()

## Orientation and area
Orientation mix and pixel-area distribution to guide resizing/augmentation.

**Why Visual 17 (grayscale histogram)**: Sampling grayscale pixels shows contrast and brightness spread, indicating if normalization or CLAHE might help. Without it, we might miss low-contrast images that hinder edge detectors and CNN learning.

In [ ]:
# Sample pixel intensities to see grayscale distribution
plt.figure(figsize=(6,4))
gray_values = []  # sampled grayscale pixels
for path in stat_sample['filepath'][:200]:  # limit to avoid huge arrays
    g = np.array(Image.open(path).convert('L')).flatten()  # grayscale pixels
    gray_values.extend(g[np.random.choice(len(g), size=min(2000, len(g)), replace=False)])  # random subset per image
sns.histplot(gray_values, bins=50, color='gray', kde=True)
plt.title('Grayscale intensity distribution (sample of pixels)')
plt.xlabel('Intensity (0-255)')
plt.ylabel('Frequency')
plt.show()

## Grayscale intensity
Histogram of sampled grayscale pixels to assess contrast/brightness spread.

**Why Visual 18 (mean RGB histograms)**: Shows distribution of per-image channel means, revealing color cast or illumination bias. Skipping it hides whether a color normalization step (e.g., per-channel standardization) is needed to stabilize training.

In [ ]:
# Distribution of per-image mean RGB channels
plt.figure(figsize=(7,4))
channel_colors = ['red','green','blue']
for ch, color in enumerate(channel_colors):
    sns.histplot(stat_df[[f'mean_{c}' for c in ['r','g','b']][ch]], bins=30, color=color, label=f'{color} mean', alpha=0.5)  # channel mean histogram
plt.title('Mean RGB values per image (sample)')
plt.xlabel('Channel mean (0-255)')
plt.ylabel('Frequency')
plt.legend()
plt.show()

## Mean RGB distribution
Channel-mean histograms to spot global color casts.

**Why Visual 19 (mean RGB per class)**: Class-level barplot checks if certain labels have systematic color differences, which might be discriminative or require balancing. Without it, we could miss color leakage cues or fail to apply augmentations that reduce over-reliance on color.

In [ ]:
# Compare average RGB levels across classes
rgb_means = stat_df.groupby('label')[['mean_r','mean_g','mean_b']].mean().reset_index()  # per-class averages
rgb_melt = rgb_means.melt(id_vars='label', var_name='channel', value_name='mean_val')  # long format for seaborn
plt.figure(figsize=(7,4))
sns.barplot(data=rgb_melt, x='label', y='mean_val', hue='channel', palette={'mean_r':'red','mean_g':'green','mean_b':'blue'})
plt.title('Mean RGB per class (sample)')
plt.xlabel('Label')
plt.ylabel('Channel mean (0-255)')
plt.xticks(rotation=20)
plt.legend(title='Channel')
plt.show()

## Per-class color means
Barplot of mean RGB per class to detect color biases.

**Why Visual 20–23 (brightness/saturation/contrast/edges by label)**: Boxplots of V, S, gray std, and Sobel edge strength highlight per-class texture/lightness differences. Skipping them hides whether some classes are darker, less saturated, or smoother—factors that can cause biased feature learning or suggest preprocessing (e.g., histogram equalization, sharpening).

In [ ]:
# Compare brightness, saturation, contrast, and edge strength per class
plt.figure(figsize=(7,4))
sns.boxplot(data=stat_df, x='label', y='brightness_v', order=sorted(stat_df['label'].unique()), palette='YlOrBr')  # V channel per class
plt.title('Brightness (V channel) by label (sample)')
plt.xlabel('Label')
plt.ylabel('Brightness (0-255)')
plt.xticks(rotation=20)
plt.show()

plt.figure(figsize=(7,4))
sns.boxplot(data=stat_df, x='label', y='saturation_s', order=sorted(stat_df['label'].unique()), palette='PuBuGn')  # saturation per class
plt.title('Saturation (S channel) by label (sample)')
plt.xlabel('Label')
plt.ylabel('Saturation (0-255)')
plt.xticks(rotation=20)
plt.show()

plt.figure(figsize=(7,4))
sns.boxplot(data=stat_df, x='label', y='contrast_gray', order=sorted(stat_df['label'].unique()), palette='cool')  # grayscale contrast per class
plt.title('Contrast (gray std) by label (sample)')
plt.xlabel('Label')
plt.ylabel('Std of gray')
plt.xticks(rotation=20)
plt.show()

plt.figure(figsize=(7,4))
sns.boxplot(data=stat_df, x='label', y='edge_strength', order=sorted(stat_df['label'].unique()), palette='rocket')  # edge strength per class
plt.title('Edge strength by label (sample)')
plt.xlabel('Label')
plt.ylabel('Mean abs Sobel response')
plt.xticks(rotation=20)
plt.show()

## Brightness, saturation, texture by label
Boxplots for V/S channels, grayscale contrast, and edge strength across classes.

**Why Visual 24 & 25 (correlation heatmap, top resolutions)**: Correlation map shows relationships among numeric features (size, color, texture), guiding feature selection or decorrelation. Top-resolution bar plot surfaces the most common shapes to pick efficient resizing targets. Without these, we might choose redundant features or an inefficient canonical resolution, wasting compute or losing detail.

In [ ]:
# Correlation across numeric features and most common resolutions
plt.figure(figsize=(8,6))
num_cols = ['width','height','area','aspect','mean_r','mean_g','mean_b','brightness_v','saturation_s','contrast_gray','edge_strength','file_kb']  # numeric feature list
corr = stat_df[num_cols].corr()  # correlation matrix
sns.heatmap(corr, cmap='coolwarm', center=0, annot=False)  # visualize correlations
plt.title('Correlation of image-level features (sample)')
plt.show()

plt.figure(figsize=(7,4))
stat_df['resolution'].value_counts().head(10).plot(kind='bar', color='teal')  # top resolutions
plt.title('Top 10 resolutions (sample)')
plt.xlabel('Resolution (WxH)')
plt.ylabel('Count')
plt.xticks(rotation=30)
plt.show()

## Feature correlations and common resolutions
Correlation heatmap of numeric features plus most frequent resolutions.

In [ ]:
# Image preprocessing: build train/val/test generators from active splitted datasets with augmentation for training and rescaling for all
#set paths for the two datasets
dataset1_path = "dataset1_70_20_10"  # path to re-split dataset 1
dataset2_path = "dataset2_70_20_10"  # path to re-split dataset 2
# Define image size and batch size for training
IMG_SIZE = (224, 224)  # standard size for many CNNs
BATCH_SIZE = 32
# Data augmentation for training set
train_datagen = ImageDataGenerator(
    rescale=1.0/255,    # rescale pixel values to [0,1]
    rotation_range=10,  # random rotations
    width_shift_range=0.1,  # horizontal shifts
    height_shift_range=0.1,  # vertical shifts
    zoom_range=0.1,  # zooms
    horizontal_flip=True,  # horizontal flips
    fill_mode='nearest'  # fill in missing pixels
)
# No data augmentation for validation and test sets
val_test_datagen = ImageDataGenerator(rescale=1.0/255)  # rescale pixel values to [0,1] for validation and test sets
# Create generators for dataset 1
train_generator1 = train_datagen.flow_from_directory(
    dataset1_path + '/train',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)
val_generator1 = val_test_datagen.flow_from_directory(
    dataset1_path + '/val',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)
test_generator1 = val_test_datagen.flow_from_directory(
    dataset1_path + '/test',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)
# Create generators for dataset 2
train_generator2 = train_datagen.flow_from_directory(
    dataset2_path + '/train',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)
val_generator2 = val_test_datagen.flow_from_directory(
    dataset2_path + '/val',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)
test_generator2 = val_test_datagen.flow_from_directory(
    dataset2_path + '/test',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False   
)
print("Data generators created successfully.")

## Model training utilities
Shared helpers to compile, train, evaluate, and plot metrics for all architectures below.

In [ ]:
# Keras training helper (shared across EfficientNet, ViT, and additional models) for both datasets with early stopping and model checkpointing
from tensorflow.keras.applications import EfficientNetB0, DenseNet121, VGG16, InceptionV3, MobileNetV2, ResNet50
from tensorflow.keras.layers import GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# Simple utility to compile, train, and evaluate a Keras model
results = {}  # store results for all models

def compile_and_train(model, label: str, epochs: int = 20):
    model.compile(
        optimizer=Adam(learning_rate=1e-4),  # Adam optimizer with a low learning rate
        loss='categorical_crossentropy',  # suitable for multi-class classification
        metrics=['accuracy']  # track accuracy during training
    )
    history = model.fit(
        train_generator,
        validation_data=val_generator,
        epochs=epochs,
        verbose=1,  # show progress bar during training
        callbacks=[
            EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),  # stop if no improvement for 3 epochs
            ModelCheckpoint(f'{label}_best_model.h5', save_best_only=True, monitor='val_loss')  # save best model based on validation loss
        ]
    )  

In [ ]:
# Utility: plot training curves for a model in `results`

def plot_history(label: str, title: str = None):
    if label not in results or "history" not in results[label]:
        print(f"No history available for {label}; run training first.")
        return
    hist = results[label]["history"]
    acc = hist.get("accuracy", hist.get("acc", []))
    val_acc = hist.get("val_accuracy", hist.get("val_acc", []))
    loss = hist.get("loss", [])
    val_loss = hist.get("val_loss", [])
    epochs_range = range(1, len(acc) + 1)

    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, acc, label="train_acc")
    plt.plot(epochs_range, val_acc, label="val_acc")
    plt.title(title or f"{label}: Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, loss, label="train_loss")
    plt.plot(epochs_range, val_loss, label="val_loss")
    plt.title(title or f"{label}: Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()

    plt.suptitle(title or label)
    plt.tight_layout()
    plt.show()


### Evaluation utilities (accuracy, precision, recall, F1, AUC)
Helpers to score trained models on the held-out test set and visualize macro-averaged metrics.

In [ ]:
# Compute test-set metrics (macro) and plot a comparison across models.

def evaluate_full_metrics(model, label: str):
    if model is None:
        print(f"{label}: model not provided.")
        return {}

    # Reset generator to ensure full sweep of test set before prediction.
    test_ds.reset()
    probs = model.predict(test_ds, verbose=0)
    preds = np.argmax(probs, axis=1)
    y_true = test_ds.classes
    class_names = [cls for cls, idx in sorted(test_ds.class_indices.items(), key=lambda x: x[1])]
    y_true_one_hot = tf.keras.utils.to_categorical(y_true, num_classes=num_classes)

    acc = float((preds == y_true).mean())
    precision = float(precision_score(y_true, preds, average="macro", zero_division=0))
    recall = float(recall_score(y_true, preds, average="macro", zero_division=0))
    f1 = float(f1_score(y_true, preds, average="macro", zero_division=0))
    try:
        auc = float(roc_auc_score(y_true_one_hot, probs, average="macro", multi_class="ovr"))
    except ValueError:
        auc = float("nan")

    metrics = {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "auc": auc,
    }

    results.setdefault(label, {}).update(
        {
            "metrics": metrics,
            "class_names": class_names,
            "y_true": y_true.tolist(),
            "y_pred": preds.tolist(),
        }
    )
    return metrics


def evaluate_and_plot(model_map: dict):
    rows = []
    for lbl, model in model_map.items():
        metrics = evaluate_full_metrics(model, lbl)
        if metrics:
            rows.append({"model": lbl, **metrics})

    if not rows:
        print("No trained models were supplied; train models first.")
        return pd.DataFrame()

    df = pd.DataFrame(rows).set_index("model")
    display(df)

    ax = df.plot(kind="bar", figsize=(10, 5), ylim=(0, 1))
    ax.set_title("Test metrics by model (macro averages)")
    ax.set_ylabel("Score (0-1)")
    ax.set_xlabel("Model")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()
    return df

## CNN baseline
Build, train, plot, and save a simple convolutional baseline to benchmark transfer models.

In [ ]:
# Define CNN baseline

# Builds a shallow convolutional classifier with batch norm regularization.
def make_cnn_baseline():
    model = Sequential(name="CNN_baseline")
    model.add(Conv2D(32, (3, 3), activation="relu", padding="same", input_shape=input_shape))
    model.add(MaxPooling2D((2, 2)))
    model.add(BatchNormalization())
    model.add(Conv2D(64, (3, 3), activation="relu", padding="same"))
    model.add(MaxPooling2D((2, 2)))
    model.add(BatchNormalization())
    model.add(Conv2D(128, (3, 3), activation="relu", padding="same"))
    model.add(MaxPooling2D((2, 2)))
    model.add(BatchNormalization())
    model.add(Flatten())
    model.add(Dense(256, activation="relu"))
    model.add(Dropout(0.4))
    model.add(Dense(num_classes, activation="softmax"))
    return model


In [ ]:
import os
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, BatchNormalization, Flatten, Dense, Dropout
from tensorflow.keras.utils import plot_model
import matplotlib.pyplot as plt
from PIL import Image

# Keep visualization settings local so training globals are not overwritten.
viz_input_shape = (128, 128, 3)
viz_num_classes = 5

# Create image directory if it does not exist
os.makedirs("images", exist_ok=True)

def make_cnn_baseline_preview():
    model = Sequential(name="CNN_baseline_preview")
    model.add(Conv2D(32, (3, 3), activation="relu", padding="same", input_shape=viz_input_shape))
    model.add(MaxPooling2D((2, 2)))
    model.add(BatchNormalization())
    model.add(Conv2D(64, (3, 3), activation="relu", padding="same"))
    model.add(MaxPooling2D((2, 2)))
    model.add(BatchNormalization())
    model.add(Conv2D(128, (3, 3), activation="relu", padding="same"))
    model.add(MaxPooling2D((2, 2)))
    model.add(BatchNormalization())
    model.add(Flatten())
    model.add(Dense(256, activation="relu"))
    model.add(Dropout(0.4))
    model.add(Dense(viz_num_classes, activation="softmax"))
    return model

cnn_preview_model = make_cnn_baseline_preview()
cnn_preview_model.build((None, *viz_input_shape))

# Save into images directory
image_path = "images/cnn_baseline_architecture.png"

try:
    plot_model(
        cnn_preview_model,
        to_file=image_path,
        show_shapes=True,
        show_layer_names=True,
        dpi=150
    )

    if os.path.exists(image_path):
        print(f"Saved successfully at: {image_path}")

        img = Image.open(image_path)
        plt.figure(figsize=(14, 10))
        plt.imshow(img)
        plt.axis("off")
        plt.show()
    else:
        print("plot_model ran, but no image file was created.")

except Exception as e:
    print("Error while generating model image:")
    print(e)

### CNN baseline — define architecture
Construct a small CNN with batch norm and dropout to act as a baseline classifier.

In [ ]:
# Train CNN baselineon dataset 1 and another on dataset2 using the compile and train method

epochs = 1
# Define globals required by make_cnn_baseline() and compile_and_train()
input_shape = (*IMG_SIZE, 3)
num_classes = train_generator1.num_classes
train_generator, val_generator, test_generator = train_generator1, val_generator1, test_generator1

cnn_model1 = make_cnn_baseline()
compile_and_train(cnn_model1, "CNN_baseline_dataset1", epochs=epochs)
cnn_model2 = make_cnn_baseline()
compile_and_train(cnn_model2, "CNN_baseline_dataset2", epochs=epochs)

# Quick view of accumulated test metrics for all trained models.
print({k: {"test_acc": round(v["test_acc"], 3), "test_loss": round(v["test_loss"], 3)} for k, v in results.items()})

### CNN baseline — train and evaluate
Instantiate the CNN, train for a placeholder epoch, and log the test metrics.

In [ ]:
# Plot CNN baseline training curves for both datasets individually to check for overfitting and convergence patterns
plot_history("CNN_baseline_dataset1", title="CNN Baseline - Dataset 1")
plot_history("CNN_baseline_dataset2", title="CNN Baseline - Dataset 2")

### CNN baseline — plot learning curves
Visualize train/val accuracy and loss for the CNN baseline.

In [ ]:
# Save CNN baseline model weights for both datasets to disk for later loading and evaluation
cnn_model1.save_weights("cnn_baseline_dataset1_weights.h5")
cnn_model2.save_weights("cnn_baseline_dataset2_weights.h5")

# Ensure models directory exists then save the baseline weights.
Path("models").mkdir(exist_ok=True)
cnn_model1.save_weights("models/cnn_baseline_dataset1_weights.h5")
cnn_model2.save_weights("models/cnn_baseline_dataset2_weights.h5")

In [ ]:
# Save CNN baseline model
from pathlib import Path

# Ensure models directory exists then save the baseline weights.
Path("models").mkdir(exist_ok=True)
cnn_model.save("models/cnn_baseline.h5")
print("Saved CNN baseline to models/cnn_baseline.h5")


### CNN baseline — save weights
Persist the trained CNN baseline to disk for later use.

## EfficientNetB0
Freeze the ImageNet backbone, add a small head, then train, plot curves, and save the model.

In [ ]:
# Define EfficientNetB0

# Use ImageNet-pretrained EfficientNetB0 as frozen feature extractor with a custom head.
def make_efficientnet_b0():
    base = EfficientNetB0(weights="imagenet", include_top=False, input_shape=input_shape)
    base.trainable = False
    x = base.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(256, activation="relu")(x)
    x = Dropout(0.3)(x)
    out = Dense(num_classes, activation="softmax")(x)
    return Model(inputs=base.input, outputs=out, name="EfficientNetB0_frozen")


### EfficientNetB0 — define architecture
Freeze EfficientNetB0 backbone and attach a small dense head for classification.

In [ ]:
# Train CNN baselineon dataset 1 and another on dataset2 using the compile and train method

epochs = 1
# Define globals required by make_cnn_baseline() and compile_and_train()
input_shape = (*IMG_SIZE, 3)
num_classes = train_generator1.num_classes
train_generator, val_generator, test_generator = train_generator1, val_generator1, test_generator1

cnn_model1 = make_cnn_baseline()
compile_and_train(cnn_model1, "make_efficientnet_b0", epochs=epochs)
cnn_model2 = make_cnn_baseline()
compile_and_train(cnn_model2, "make_efficientnet_b0", epochs=epochs)

# Quick view of accumulated test metrics for all trained models.
print({k: {"test_acc": round(v["test_acc"], 3), "test_loss": round(v["test_loss"], 3)} for k, v in results.items()})

### EfficientNetB0 — train and evaluate
Instantiate EfficientNetB0 classifier, run placeholder epoch, and log metrics.

In [ ]:
# Plot EfficientNetB0 training curves
# Helper plots accuracy and loss across epochs.
plot_history("efficientnet_b0", title="EfficientNetB0")


### EfficientNetB0 — plot learning curves
Use shared plotting helper to visualize EfficientNetB0 training history.

## DenseNet121
Use a frozen DenseNet121 feature extractor, fine-tune the classifier head, visualize training, and save.

In [ ]:
# Define DenseNet121

# Use ImageNet-pretrained DenseNet121 frozen as feature extractor with custom classifier head.
def make_densenet121():
    base = DenseNet121(weights="imagenet", include_top=False, input_shape=input_shape)
    base.trainable = False
    x = base.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(256, activation="relu")(x)
    x = Dropout(0.3)(x)
    out = Dense(num_classes, activation="softmax")(x)
    return Model(inputs=base.input, outputs=out, name="DenseNet121_frozen")


### DenseNet121 — define architecture
Freeze DenseNet121 features and attach a compact classifier head.

In [ ]:
# Train DenseNet121
# Placeholder epochs value; increase for full runs.
epochs = 1
densenet_model = make_densenet121()
densenet_results = compile_and_train(densenet_model, "densenet121", epochs=epochs)

# Display summary metrics accumulated so far.
print({k: {"test_acc": round(v["test_acc"], 3), "test_loss": round(v["test_loss"], 3)} for k, v in results.items()})


### DenseNet121 — train and evaluate
Create the DenseNet121 model, run one epoch placeholder training, and record metrics.

In [ ]:
# Plot DenseNet121 training curves
# Visualize training and validation metrics for DenseNet121.
plot_history("densenet121", title="DenseNet121")


### DenseNet121 — plot learning curves
Plot accuracy and loss for the DenseNet121 run via the shared helper.

In [ ]:
# Save DenseNet121 model
from pathlib import Path

# Persist the trained DenseNet121 weights.
Path("models").mkdir(exist_ok=True)
densenet_model.save("models/densenet121.h5")
print("Saved DenseNet121 to models/densenet121.h5")


### DenseNet121 — save weights
Store the DenseNet121 model after training.

## VGG16
Train a frozen VGG16 backbone with a lightweight head, plot accuracy/loss, and persist the weights.

In [ ]:
# Define VGG16

# Use pretrained VGG16 frozen as a feature extractor with a small dense head.
def make_vgg16():
    base = VGG16(weights="imagenet", include_top=False, input_shape=input_shape)
    base.trainable = False
    x = base.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(256, activation="relu")(x)
    x = Dropout(0.3)(x)
    out = Dense(num_classes, activation="softmax")(x)
    return Model(inputs=base.input, outputs=out, name="VGG16_frozen")


### VGG16 — define architecture
Freeze VGG16 convolutional base and attach a custom dense classifier.

In [ ]:
# Train VGG16
# Placeholder epochs value; increase when ready to train longer.
epochs = 1
vgg16_model = make_vgg16()
vgg16_results = compile_and_train(vgg16_model, "vgg16", epochs=epochs)

# Report metrics across all trained models.
print({k: {"test_acc": round(v["test_acc"], 3), "test_loss": round(v["test_loss"], 3)} for k, v in results.items()})


### VGG16 — train and evaluate
Instantiate VGG16 classifier, run placeholder epoch, and capture metrics.

In [ ]:
# Plot VGG16 training curves
# Visualize training/validation accuracy and loss for VGG16.
plot_history("vgg16", title="VGG16")


### VGG16 — plot learning curves
Plot train/val accuracy and loss for the VGG16 run.

In [ ]:
# Save VGG16 model
from pathlib import Path

# Persist VGG16 weights for later inference.
Path("models").mkdir(exist_ok=True)
vgg16_model.save("models/vgg16.h5")
print("Saved VGG16 to models/vgg16.h5")


### VGG16 — save weights
Save the trained VGG16 model to disk.

## InceptionV3
Leverage InceptionV3 features with a custom head, track training curves, and save the model.

In [ ]:
# Define InceptionV3

# Use pretrained InceptionV3 frozen as feature extractor with a dense head.
def make_inception_v3():
    base = InceptionV3(weights="imagenet", include_top=False, input_shape=input_shape)
    base.trainable = False
    x = base.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(256, activation="relu")(x)
    x = Dropout(0.3)(x)
    out = Dense(num_classes, activation="softmax")(x)
    return Model(inputs=base.input, outputs=out, name="InceptionV3_frozen")


### InceptionV3 — define architecture
Freeze InceptionV3 base and add a compact classifier head.

In [ ]:
# Train InceptionV3
# Placeholder epochs value; increase later for better convergence.
epochs = 1
inception_model = make_inception_v3()
inception_results = compile_and_train(inception_model, "inception_v3", epochs=epochs)

# Display current metrics table.
print({k: {"test_acc": round(v["test_acc"], 3), "test_loss": round(v["test_loss"], 3)} for k, v in results.items()})


### InceptionV3 — train and evaluate
Instantiate InceptionV3 classifier, run one-epoch placeholder training, and print metrics.

In [ ]:
# Plot InceptionV3 training curves
# Visualize train/validation metrics for InceptionV3.
plot_history("inception_v3", title="InceptionV3")


### InceptionV3 — plot learning curves
Plot accuracy/loss for the InceptionV3 run using the shared helper.

In [ ]:
# Save InceptionV3 model
from pathlib import Path

# Persist trained InceptionV3 weights.
Path("models").mkdir(exist_ok=True)
inception_model.save("models/inception_v3.h5")
print("Saved InceptionV3 to models/inception_v3.h5")


### InceptionV3 — save weights
Save the trained InceptionV3 model to disk.

## MobileNetV2
Train a lightweight MobileNetV2 classifier head, inspect learning curves, and save for reuse.

In [ ]:
# Define MobileNetV2

# Use pretrained MobileNetV2 frozen as feature extractor with a dense classification head.
def make_mobilenet_v2():
    base = MobileNetV2(weights="imagenet", include_top=False, input_shape=input_shape)
    base.trainable = False
    x = base.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(256, activation="relu")(x)
    x = Dropout(0.3)(x)
    out = Dense(num_classes, activation="softmax")(x)
    return Model(inputs=base.input, outputs=out, name="MobileNetV2_frozen")


### MobileNetV2 — define architecture
Freeze MobileNetV2 backbone and attach a lightweight dense head.

## ResNet50
Train and evaluate a frozen ResNet50 backbone with a dense head, plot performance, and save the artifact.

### ResNet50 — define architecture
Freeze ResNet50 backbone and add a dense head for classification.

In [ ]:
# Define ResNet50

# Use pretrained ResNet50 frozen as feature extractor with a dense head.
def make_resnet50():
    base = ResNet50(weights="imagenet", include_top=False, input_shape=input_shape)
    base.trainable = False
    x = base.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(256, activation="relu")(x)
    x = Dropout(0.3)(x)
    out = Dense(num_classes, activation="softmax")(x)
    return Model(inputs=base.input, outputs=out, name="ResNet50_frozen")


In [ ]:
# Train MobileNetV2
# Placeholder epochs; increase for proper training.
epochs = 1
mobilenet_model = make_mobilenet_v2()
mobilenet_results = compile_and_train(mobilenet_model, "mobilenet_v2", epochs=epochs)

# Print consolidated metrics across models.
print({k: {"test_acc": round(v["test_acc"], 3), "test_loss": round(v["test_loss"], 3)} for k, v in results.items()})


### MobileNetV2 — train and evaluate
Instantiate MobileNetV2 classifier, run one epoch, and show metrics.

In [ ]:
# Train ResNet50
# Placeholder epochs value; bump up when ready for longer training.
epochs = 1
resnet_model = make_resnet50()
resnet_results = compile_and_train(resnet_model, "resnet50", epochs=epochs)

# Display cumulative metrics across models.
print({k: {"test_acc": round(v["test_acc"], 3), "test_loss": round(v["test_loss"], 3)} for k, v in results.items()})


### ResNet50 — train and evaluate
Instantiate ResNet50 classifier, run one epoch placeholder, and log metrics.

In [ ]:
# Plot MobileNetV2 training curves
# Visualize MobileNetV2 accuracy and loss progression.
plot_history("mobilenet_v2", title="MobileNetV2")


### MobileNetV2 — plot learning curves
Plot training/validation accuracy and loss for MobileNetV2.

In [ ]:
# Save MobileNetV2 model
from pathlib import Path

# Persist MobileNetV2 weights for reuse.
Path("models").mkdir(exist_ok=True)
mobilenet_model.save("models/mobilenet_v2.h5")
print("Saved MobileNetV2 to models/mobilenet_v2.h5")


### MobileNetV2 — save weights
Save the trained MobileNetV2 model to disk.

In [ ]:
# Plot ResNet50 training curves
# Visualize training/validation metrics for ResNet50.
plot_history("resnet50", title="ResNet50")


### ResNet50 — plot learning curves
Plot accuracy and loss for the ResNet50 run using the helper.

In [ ]:
# Visualize EfficientNetB0 training curves
# Manual plotting (redundant with plot_history) for flexibility.
if "efficientnet_b0" in results and "history" in results["efficientnet_b0"]:
    hist = results["efficientnet_b0"]["history"]
    acc = hist.get("accuracy", hist.get("acc", []))
    val_acc = hist.get("val_accuracy", hist.get("val_acc", []))
    loss = hist.get("loss", [])
    val_loss = hist.get("val_loss", [])
    epochs_range = range(1, len(acc) + 1)

    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, acc, label="train_acc")
    plt.plot(epochs_range, val_acc, label="val_acc")
    plt.title("EfficientNetB0: Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, loss, label="train_loss")
    plt.plot(epochs_range, val_loss, label="val_loss")
    plt.title("EfficientNetB0: Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()

    plt.suptitle("Training curves: EfficientNetB0")
    plt.tight_layout()
    plt.show()
else:
    print("No history available for EfficientNetB0; run training first.")


### EfficientNetB0 — manual plot (optional)
Manual Matplotlib plotting of EfficientNetB0 accuracy/loss (redundant with helper above).

In [ ]:
# Save EfficientNetB0 Keras model
from pathlib import Path

# Persist EfficientNetB0 weights.
Path("models").mkdir(exist_ok=True)
EffNetB0_model.save("models/efficientnet_b0.h5")
print("Saved EfficientNetB0 to models/efficientnet_b0.h5")


### EfficientNetB0 — save weights
Save the trained EfficientNetB0 model to disk.

In [ ]:
# Save ResNet50 model
from pathlib import Path

# Persist ResNet50 weights for later inference.
Path("models").mkdir(exist_ok=True)
resnet_model.save("models/resnet50.h5")
print("Saved ResNet50 to models/resnet50.h5")


### ResNet50 — save weights
Save the trained ResNet50 model to disk.

## Vision Transformer (Keras)
Build, train, plot, and save a compact ViT classifier to mirror the same workflow used by other models.

In [ ]:
# Define Vision Transformer (Keras)
from tensorflow.keras import layers

# Builds a compact Vision Transformer classifier with patch embedding and transformer blocks.
def make_vit_keras():
    patch_size = 16
    projection_dim = 64
    transformer_layers = 6
    num_heads = 4
    dropout_rate = 0.1
    num_patches = (input_shape[0] // patch_size) * (input_shape[1] // patch_size)

    def vit_mlp(x, hidden_units, dropout):
        for units in hidden_units:
            x = layers.Dense(units, activation="gelu")(x)
            x = layers.Dropout(dropout)(x)
        return x

    inputs = layers.Input(shape=input_shape)
    x = layers.Conv2D(filters=projection_dim, kernel_size=patch_size, strides=patch_size, padding="valid")(inputs)
    x = layers.Reshape((num_patches, projection_dim))(x)

    positions = tf.range(start=0, limit=num_patches, delta=1)
    pos_embedding = layers.Embedding(input_dim=num_patches, output_dim=projection_dim)(positions)
    x = x + pos_embedding

    for _ in range(transformer_layers):
        x1 = layers.LayerNormalization(epsilon=1e-6)(x)
        attn_output = layers.MultiHeadAttention(num_heads=num_heads, key_dim=projection_dim, dropout=dropout_rate)(x1, x1)
        x2 = layers.Add()([attn_output, x])
        x3 = layers.LayerNormalization(epsilon=1e-6)(x2)
        x3 = vit_mlp(x3, hidden_units=[projection_dim * 2, projection_dim], dropout=dropout_rate)
        x = layers.Add()([x3, x2])

    x = layers.LayerNormalization(epsilon=1e-6)(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(dropout_rate)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    return Model(inputs=inputs, outputs=outputs, name="ViT_Keras")


### Vision Transformer — define architecture
Define a compact ViT with patch embedding, transformer encoder blocks, and a softmax classifier head.

In [ ]:
# Train Vision Transformer (Keras)
# Set placeholder epochs; increase later for real training.
epochs = 1
vit_keras_model = make_vit_keras()
vit_results = compile_and_train(vit_keras_model, "vit_keras", epochs=epochs)

# Quick view of accumulated test metrics for all trained models.
print({k: {"test_acc": round(v["test_acc"], 3), "test_loss": round(v["test_loss"], 3)} for k, v in results.items()})


### Vision Transformer — train and evaluate
Instantiate the ViT model, run placeholder training, and log the test metrics.

In [ ]:
# Plot Vision Transformer training curves
# Uses shared helper to draw accuracy/loss for this run.
plot_history("vit_keras", title="ViT (Keras)")


### Vision Transformer — plot learning curves
Use the shared plotting helper to visualize train/validation accuracy and loss for ViT.

In [ ]:
# Save Keras ViT model
from pathlib import Path

# Ensure models directory exists then save the ViT weights.
Path("models").mkdir(exist_ok=True)
vit_keras_model.save("models/vit_keras.h5")
print("Saved ViT (Keras) to models/vit_keras.h5")


### Vision Transformer — save weights
Save the trained ViT model to disk.

## Compare trained models on test metrics
Run this after training to visualize accuracy, precision, recall, F1, and macro AUC across available models.

In [ ]:
# Gather trained models in the current session and visualize macro metrics.
trained_models = {}
for name, var in [
    ("cnn_baseline", "cnn_model"),
    ("efficientnet_b0", "EffNetB0_model"),
    ("densenet121", "densenet_model"),
    ("vgg16", "vgg16_model"),
    ("inception_v3", "inception_model"),
    ("mobilenet_v2", "mobilenet_model"),
    ("resnet50", "resnet_model"),
    ("vit_keras", "vit_keras_model"),
]:
    if var in globals():
        trained_models[name] = globals()[var]

metrics_df = evaluate_and_plot(trained_models)

In [ ]:
import os
os.environ["PATH"] += os.pathsep + r"C:\Program Files\Graphviz\bin"

## IEEE architecture visualizations for project models
Generate publication-ready architecture diagrams for all notebook models.

Exports:
- `architectures-images/*.png` (300 DPI raster)

This cell expects the model builder functions to already be defined.

In [ ]:
# =============================================================================
# IEEE-COMPLIANT ARCHITECTURE VISUALIZATION (INTERLINKED + PUBLICATION-GRADE)
# =============================================================================

from pathlib import Path
import os
import shutil
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from PIL import Image
import tensorflow as tf
from tensorflow.keras.utils import plot_model

print("\n" + "=" * 80)
print(" IEEE PUBLICATION-GRADE ARCHITECTURE GENERATION (ADVANCED)")
print("=" * 80)

# =============================================================================
# OUTPUT DIRECTORY
# =============================================================================

output_base = Path("architectures-images")
output_base.mkdir(parents=True, exist_ok=True)
print(f"\nOutput directory: {output_base.resolve()}")

# =============================================================================
# GRAPHVIZ RUNTIME CHECK (IMPORTANT)
# =============================================================================

def ensure_graphviz_in_path():
    dot_path = shutil.which("dot")
    if dot_path:
        return dot_path

    candidates = [
        Path(r"C:\Program Files\Graphviz\bin"),
        Path(r"C:\Program Files (x86)\Graphviz2.38\bin"),
    ]

    for c in candidates:
        dot_exe = c / "dot.exe"
        if dot_exe.exists():
            os.environ["PATH"] += os.pathsep + str(c)
            return str(dot_exe)

    return None

DOT_EXE = ensure_graphviz_in_path()
print("Graphviz dot:", DOT_EXE if DOT_EXE else "NOT FOUND in current kernel PATH")

# =============================================================================
# MODEL DEFINITIONS
# =============================================================================

model_specs = [
    ("CNN Baseline", "make_cnn_baseline", Path("models/cnn_baseline.h5"), "fig_cnn_baseline_arch"),
    ("EfficientNetB0", "make_efficientnet_b0", Path("models/efficientnet_b0.h5"), "fig_efficientnet_b0_arch"),
    ("DenseNet121", "make_densenet121", Path("models/densenet121.h5"), "fig_densenet121_arch"),
    ("VGG16", "make_vgg16", Path("models/vgg16.h5"), "fig_vgg16_arch"),
    ("InceptionV3", "make_inception_v3", Path("models/inception_v3.h5"), "fig_inception_v3_arch"),
    ("MobileNetV2", "make_mobilenet_v2", Path("models/mobilenet_v2.h5"), "fig_mobilenet_v2_arch"),
    ("ResNet50", "make_resnet50", Path("models/resnet50.h5"), "fig_resnet50_arch"),
    ("Vision Transformer (Keras)", "make_vit_keras", Path("models/vit_keras.h5"), "fig_vit_keras_arch"),
]

# =============================================================================
# IEEE SEMANTIC ARCHITECTURES WITH CONNECTIONS
# =============================================================================

IEEE_ARCHS = {
    "CNN Baseline": {
        "stages": [
            {"title": "Input", "desc": "224x224x3"},
            {"title": "Feature Extraction", "desc": "Conv Blocks"},
            {"title": "Pooling", "desc": "Downsampling"},
            {"title": "Classifier", "desc": "Dense Layers"},
            {"title": "Output", "desc": "Softmax"},
        ],
        "connections": [(0, 1), (1, 2), (2, 3), (3, 4)],
    },
    "EfficientNetB0": {
        "stages": [
            {"title": "Input", "desc": "224x224x3"},
            {"title": "Stem Conv", "desc": "Initial conv"},
            {"title": "MBConv Stack", "desc": "Depthwise blocks"},
            {"title": "Global Pool", "desc": "Feature aggregate"},
            {"title": "Dense Head", "desc": "Classifier"},
            {"title": "Output", "desc": "Softmax"},
        ],
        "connections": [(0, 1), (1, 2), (2, 3), (3, 4), (4, 5)],
    },
    "DenseNet121": {
        "stages": [
            {"title": "Input", "desc": "224x224x3"},
            {"title": "Conv Stem", "desc": "Initial conv/pool"},
            {"title": "Dense Blocks", "desc": "Feature reuse"},
            {"title": "Transition", "desc": "Compression"},
            {"title": "Global Pool", "desc": "Aggregation"},
            {"title": "Dense Head", "desc": "Classifier"},
            {"title": "Output", "desc": "Softmax"},
        ],
        "connections": [(0, 1), (1, 2), (2, 3), (3, 4), (4, 5), (5, 6), (1, 3), (2, 4)],
    },
    "VGG16": {
        "stages": [
            {"title": "Input", "desc": "224x224x3"},
            {"title": "Conv Blocks", "desc": "3x3 convolutions"},
            {"title": "Max Pooling", "desc": "Downsampling"},
            {"title": "Global Pool", "desc": "Feature aggregate"},
            {"title": "Dense Head", "desc": "Classifier"},
            {"title": "Output", "desc": "Softmax"},
        ],
        "connections": [(0, 1), (1, 2), (2, 3), (3, 4), (4, 5)],
    },
    "InceptionV3": {
        "stages": [
            {"title": "Input", "desc": "224x224x3"},
            {"title": "Stem", "desc": "Initial conv stack"},
            {"title": "Inception Modules", "desc": "Multi-branch conv"},
            {"title": "Reduction", "desc": "Spatial downsample"},
            {"title": "Global Pool", "desc": "Aggregation"},
            {"title": "Dense Head", "desc": "Classifier"},
            {"title": "Output", "desc": "Softmax"},
        ],
        "connections": [(0, 1), (1, 2), (2, 3), (3, 4), (4, 5), (5, 6), (1, 3)],
    },
    "MobileNetV2": {
        "stages": [
            {"title": "Input", "desc": "224x224x3"},
            {"title": "Stem Conv", "desc": "Initial conv"},
            {"title": "Inverted Residual", "desc": "Bottleneck blocks"},
            {"title": "Linear Bottleneck", "desc": "Projection"},
            {"title": "Global Pool", "desc": "Aggregation"},
            {"title": "Dense Head", "desc": "Classifier"},
            {"title": "Output", "desc": "Softmax"},
        ],
        "connections": [(0, 1), (1, 2), (2, 3), (3, 4), (4, 5), (5, 6), (1, 3)],
    },
    "ResNet50": {
        "stages": [
            {"title": "Input", "desc": "224x224x3"},
            {"title": "Conv Stem", "desc": "Initial conv"},
            {"title": "Residual Block 1", "desc": "Skip"},
            {"title": "Residual Block 2", "desc": "Skip"},
            {"title": "Global Pool", "desc": "Aggregation"},
            {"title": "Output", "desc": "Softmax"},
        ],
        "connections": [(0, 1), (1, 2), (2, 3), (3, 4), (4, 5), (1, 3), (2, 4)],
    },
    "Vision Transformer (Keras)": {
        "stages": [
            {"title": "Input", "desc": "224x224x3"},
            {"title": "Patch Embedding", "desc": "16x16 patches"},
            {"title": "Positional Encoding", "desc": "Token positions"},
            {"title": "Transformer Encoder", "desc": "MHA + MLP"},
            {"title": "Global Pool", "desc": "Token aggregate"},
            {"title": "MLP Head", "desc": "Dense classifier"},
            {"title": "Output", "desc": "Softmax"},
        ],
        "connections": [(0, 1), (1, 2), (2, 3), (3, 4), (4, 5), (5, 6)],
    },
}

# =============================================================================
# ADVANCED IEEE RENDERER (WITH CURVED ARROWS)
# =============================================================================
def draw_ieee_architecture(stages, connections, title, png_path):
    fig, ax = plt.subplots(figsize=(20, 6))
    ax.axis("off")

    colors = ["#CFE2F3", "#D9EAD3", "#EAD1DC", "#FFF2CC", "#F4CCCC", "#D0E0E3", "#FCE5CD"]

    positions = {}
    x_start = 1
    y = 3
    width = 2.4
    height = 1.5
    spacing = 3.1

    for i, stage in enumerate(stages):
        x = x_start + i * spacing
        positions[i] = (x, y)

        rect = FancyBboxPatch(
            (x, y),
            width,
            height,
            boxstyle="round,pad=0.03",
            linewidth=1.5,
            edgecolor="black",
            facecolor=colors[i % len(colors)],
            zorder=1,
        )
        ax.add_patch(rect)

        ax.text(
            x + width / 2,
            y + height * 0.65,
            stage["title"],
            ha="center",
            fontsize=10.5,
            fontweight="bold",
            zorder=2,
        )

        ax.text(
            x + width / 2,
            y + height * 0.35,
            stage["desc"],
            ha="center",
            fontsize=8.8,
            zorder=2,
        )

    for (i, j) in connections:
        x1, y1 = positions[i]
        x2, y2 = positions[j]

        start = (x1 + width + 0.08, y1 + height / 2)
        end = (x2 - 0.08, y2 + height / 2)
        dx = j - i

        if abs(dx) == 1:
            ax.annotate(
                "",
                xy=end,
                xytext=start,
                arrowprops=dict(arrowstyle="->", lw=2.0, color="black"),
                zorder=3,
            )
        else:
            arrow = FancyArrowPatch(
                posA=(x1 + width / 2, y1 + height + 0.2),
                posB=(x2 + width / 2, y2 + height + 0.2),
                connectionstyle="arc3,rad=0.35",
                arrowstyle="->",
                linewidth=1.8,
                color="black",
                zorder=3,
            )
            ax.add_patch(arrow)

    ax.set_xlim(0, x_start + len(stages) * spacing + 2)
    ax.set_ylim(1, 6)

    plt.title(title, fontsize=14, fontweight="bold")
    plt.savefig(png_path, dpi=320, bbox_inches="tight")
    plt.close()

# =============================================================================
# MODEL RESOLUTION
# =============================================================================

def resolve_model(model_name, builder_name, fallback_path):
    if builder_name in globals() and callable(globals()[builder_name]):
        try:
            return globals()[builder_name](), f"builder:{builder_name}"
        except Exception:
            pass

    if fallback_path.exists():
        return tf.keras.models.load_model(fallback_path, compile=False), f"saved:{fallback_path.name}"

    raise ValueError(f"Model not found: {model_name}")

# =============================================================================
# EXPORT FUNCTION
# =============================================================================

def export_architecture_png(model_name, model, base_filename):
    png_path = output_base / f"{base_filename}.png"

    if model_name in IEEE_ARCHS:
        arch = IEEE_ARCHS[model_name]
        draw_ieee_architecture(
            arch["stages"],
            arch["connections"],
            f"{model_name} Architecture",
            png_path,
        )
        return png_path, "ieee_interlinked"

    if not DOT_EXE:
        return png_path, "failed_no_dot"

    try:
        plot_model(
            model,
            to_file=str(png_path),
            show_shapes=True,
            show_layer_names=True,
            expand_nested=False,
            rankdir="LR",
            dpi=220,
        )
        return png_path, "keras_plot_LR"
    except Exception as e:
        return png_path, f"failed:{e}"

# =============================================================================
# GENERATION LOOP
# =============================================================================

print("\nGenerating IEEE-quality diagrams with interconnections...\n")

generated_files = []

for i, (model_name, builder_name, fallback_path, base_filename) in enumerate(model_specs, 1):
    print(f"{i}. {model_name}")

    try:
        model = None
        source = "semantic"
        if model_name not in IEEE_ARCHS:
            model, source = resolve_model(model_name, builder_name, fallback_path)

        png_path, mode = export_architecture_png(model_name, model, base_filename)

        if png_path.exists():
            size_kb = png_path.stat().st_size / 1024
            generated_files.append(
                {
                    "model": model_name,
                    "file": png_path.name,
                    "size_kb": round(size_kb, 1),
                    "mode": mode,
                    "source": source,
                }
            )
            print(f"   Saved: {png_path.name} ({size_kb:.1f} KB)")
            print(f"   Mode: {mode}")
            print(f"   Source: {source}")
        else:
            print(f"   ERROR: Output missing after export ({mode})")

    except Exception as e:
        print(f"   ERROR: {e}")

# =============================================================================
# SUMMARY
# =============================================================================

print("\n" + "=" * 80)
print(" GENERATION COMPLETE")
print("=" * 80)

for f in generated_files:
    print(f"{f['model']} -> {f['file']} ({f['mode']}, {f['source']})")

# =============================================================================
# PREVIEW
# =============================================================================

if generated_files:
    sample = output_base / generated_files[0]["file"]
    if sample.exists():
        with Image.open(sample) as img:
            plt.figure(figsize=(14, 6))
            plt.imshow(img)
            plt.axis("off")
            plt.title("Sample IEEE Architecture Output (Interlinked)")
            plt.show()

## Unified multi-dataset training (dataset1 + dataset2)
Run the same preprocessing/training logic for both datasets, train one model per dataset, and generate concurrent comparison visuals.

In [ ]:
# Unified dual-dataset pipeline: same logic for dataset1 and dataset2 with separate models
from pathlib import Path
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, BatchNormalization, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam

# Prefer re-split folders if they exist; otherwise use original dataset folders.
DATASET_CONFIGS = {
    "dataset1": Path("dataset1_70_20_10") if Path("dataset1_70_20_10").exists() else Path("dataset1"),
    "dataset2": Path("dataset2_70_20_10") if Path("dataset2_70_20_10").exists() else Path("dataset2"),
}

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS_PER_DATASET = 10
LEARNING_RATE = 1e-4
MODEL_DIR = Path("models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)


def build_cnn(input_shape, num_classes):
    model = Sequential(name="cnn_baseline")
    model.add(Conv2D(32, (3, 3), activation="relu", padding="same", input_shape=input_shape))
    model.add(MaxPooling2D((2, 2)))
    model.add(BatchNormalization())

    model.add(Conv2D(64, (3, 3), activation="relu", padding="same"))
    model.add(MaxPooling2D((2, 2)))
    model.add(BatchNormalization())

    model.add(Conv2D(128, (3, 3), activation="relu", padding="same"))
    model.add(MaxPooling2D((2, 2)))
    model.add(BatchNormalization())

    model.add(Flatten())
    model.add(Dense(256, activation="relu"))
    model.add(Dropout(0.4))
    model.add(Dense(num_classes, activation="softmax"))
    return model


def make_generators(dataset_root, img_size=IMG_SIZE, batch_size=BATCH_SIZE):
    train_dir = dataset_root / "train"
    val_dir = dataset_root / "val"
    test_dir = dataset_root / "test"

    if not train_dir.exists() or not val_dir.exists() or not test_dir.exists():
        raise FileNotFoundError(f"Missing split folders under {dataset_root}")

    augment = ImageDataGenerator(
        rescale=1.0 / 255,
        rotation_range=10,
        width_shift_range=0.05,
        height_shift_range=0.05,
        zoom_range=0.10,
        horizontal_flip=True,
        fill_mode="nearest",
    )
    plain = ImageDataGenerator(rescale=1.0 / 255)

    train_ds = augment.flow_from_directory(
        train_dir,
        target_size=img_size,
        batch_size=batch_size,
        class_mode="categorical",
        shuffle=True,
    )
    val_ds = plain.flow_from_directory(
        val_dir,
        target_size=img_size,
        batch_size=batch_size,
        class_mode="categorical",
        shuffle=False,
    )
    test_ds = plain.flow_from_directory(
        test_dir,
        target_size=img_size,
        batch_size=batch_size,
        class_mode="categorical",
        shuffle=False,
    )
    return train_ds, val_ds, test_ds


def count_split_images(dataset_root):
    rows = []
    for split in ["train", "val", "test"]:
        split_dir = dataset_root / split
        if not split_dir.exists():
            continue
        for cls in sorted([p.name for p in split_dir.iterdir() if p.is_dir()]):
            rows.append(
                {
                    "split": split,
                    "label": cls,
                    "count": len([x for x in (split_dir / cls).iterdir() if x.is_file()]),
                }
            )
    return pd.DataFrame(rows)


multi_dataset_runs = {}
all_split_counts = {}
summary_rows = []

for dataset_name, dataset_root in DATASET_CONFIGS.items():
    print("\n" + "=" * 80)
    print(f"Running pipeline for: {dataset_name} ({dataset_root})")
    print("=" * 80)

    train_ds_local, val_ds_local, test_ds_local = make_generators(dataset_root)

    input_shape_local = IMG_SIZE + (3,)
    num_classes_local = len(train_ds_local.class_indices)

    model_local = build_cnn(input_shape_local, num_classes_local)
    model_local.compile(
        optimizer=Adam(learning_rate=LEARNING_RATE),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )

    history_local = model_local.fit(
        train_ds_local,
        validation_data=val_ds_local,
        epochs=EPOCHS_PER_DATASET,
        verbose=1,
    )

    test_loss_local, test_acc_local = model_local.evaluate(test_ds_local, verbose=0)
    model_path = MODEL_DIR / f"cnn_baseline_{dataset_name}.h5"
    model_local.save(model_path)

    print(f"{dataset_name} -> test_acc={test_acc_local:.4f}, test_loss={test_loss_local:.4f}")
    print(f"Saved model: {model_path}")

    multi_dataset_runs[dataset_name] = {
        "dataset_root": str(dataset_root),
        "model": model_local,
        "history": history_local.history,
        "test_acc": float(test_acc_local),
        "test_loss": float(test_loss_local),
        "class_indices": train_ds_local.class_indices,
        "model_path": str(model_path),
    }

    all_split_counts[dataset_name] = count_split_images(dataset_root)

    summary_rows.append(
        {
            "dataset": dataset_name,
            "classes": num_classes_local,
            "train_samples": int(train_ds_local.samples),
            "val_samples": int(val_ds_local.samples),
            "test_samples": int(test_ds_local.samples),
            "test_accuracy": float(test_acc_local),
            "test_loss": float(test_loss_local),
        }
    )

summary_df = pd.DataFrame(summary_rows).sort_values("dataset")
print("\nTraining summary:")
display(summary_df)

# Concurrent visual 1: class distribution per dataset (train split only)
fig, axes = plt.subplots(1, len(DATASET_CONFIGS), figsize=(7 * len(DATASET_CONFIGS), 5), squeeze=False)
for idx, dataset_name in enumerate(DATASET_CONFIGS.keys()):
    ax = axes[0, idx]
    counts_df = all_split_counts[dataset_name]
    train_counts_df = counts_df[counts_df["split"] == "train"]
    sns.barplot(data=train_counts_df, x="label", y="count", ax=ax)
    ax.set_title(f"{dataset_name}: train class distribution")
    ax.set_xlabel("Class")
    ax.set_ylabel("Images")
    ax.tick_params(axis="x", rotation=25)
plt.tight_layout()
plt.show()

# Concurrent visual 2: training curves side-by-side (accuracy and loss)
fig, axes = plt.subplots(2, len(DATASET_CONFIGS), figsize=(7 * len(DATASET_CONFIGS), 8), squeeze=False)
for idx, dataset_name in enumerate(DATASET_CONFIGS.keys()):
    hist = multi_dataset_runs[dataset_name]["history"]
    epochs_axis = range(1, len(hist.get("accuracy", [])) + 1)

    axes[0, idx].plot(epochs_axis, hist.get("accuracy", []), label="train_acc")
    axes[0, idx].plot(epochs_axis, hist.get("val_accuracy", []), label="val_acc")
    axes[0, idx].set_title(f"{dataset_name}: accuracy")
    axes[0, idx].set_xlabel("Epoch")
    axes[0, idx].set_ylabel("Accuracy")
    axes[0, idx].legend()

    axes[1, idx].plot(epochs_axis, hist.get("loss", []), label="train_loss")
    axes[1, idx].plot(epochs_axis, hist.get("val_loss", []), label="val_loss")
    axes[1, idx].set_title(f"{dataset_name}: loss")
    axes[1, idx].set_xlabel("Epoch")
    axes[1, idx].set_ylabel("Loss")
    axes[1, idx].legend()

plt.tight_layout()
plt.show()

# Concurrent visual 3: cross-dataset test accuracy comparison
plt.figure(figsize=(7, 4))
sns.barplot(data=summary_df, x="dataset", y="test_accuracy")
plt.title("Test accuracy comparison across datasets")
plt.ylim(0, 1)
plt.xlabel("Dataset")
plt.ylabel("Accuracy")
plt.tight_layout()
plt.show()

## Concurrent EDA parity (dataset1 vs dataset2)
Replicates the key EDA visuals for both datasets in synchronized side-by-side plots: class/split balance, geometry, color statistics, texture proxies, and common resolutions.

In [ ]:
# Concurrent EDA parity block for dataset1 and dataset2
from pathlib import Path
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from PIL import Image
import cv2

sns.set_style("darkgrid")

EDA_DATASETS = {
    "dataset1": Path("dataset1_70_20_10") if Path("dataset1_70_20_10").exists() else Path("dataset1"),
    "dataset2": Path("dataset2_70_20_10") if Path("dataset2_70_20_10").exists() else Path("dataset2"),
}


def build_eda_df(dataset_root: Path) -> pd.DataFrame:
    rows = []
    for split in ["train", "val", "test"]:
        split_dir = dataset_root / split
        if not split_dir.exists():
            continue
        for label_dir in sorted([p for p in split_dir.iterdir() if p.is_dir()]):
            for img_path in label_dir.glob("*"):
                if img_path.is_file():
                    rows.append(
                        {
                            "split": split,
                            "label": label_dir.name,
                            "filepath": str(img_path),
                        }
                    )
    return pd.DataFrame(rows)


def build_stat_df(eda_df: pd.DataFrame, sample_size: int = 400) -> pd.DataFrame:
    if eda_df.empty:
        return pd.DataFrame()

    sample = eda_df.sample(min(len(eda_df), sample_size), random_state=42).reset_index(drop=True)
    stats = []
    for _, row in sample.iterrows():
        arr = np.array(Image.open(row["filepath"]).convert("RGB"))
        h, w, _ = arr.shape
        gray = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)
        hsv = cv2.cvtColor(arr, cv2.COLOR_RGB2HSV)
        sobel = cv2.Sobel(gray, cv2.CV_64F, 1, 1, ksize=3)

        stats.append(
            {
                "split": row["split"],
                "label": row["label"],
                "width": w,
                "height": h,
                "area": w * h,
                "aspect": w / h,
                "orientation": "landscape" if w > h else ("portrait" if h > w else "square"),
                "mean_r": arr[:, :, 0].mean(),
                "mean_g": arr[:, :, 1].mean(),
                "mean_b": arr[:, :, 2].mean(),
                "brightness_v": hsv[:, :, 2].mean(),
                "saturation_s": hsv[:, :, 1].mean(),
                "contrast_gray": gray.std(),
                "edge_strength": np.mean(np.abs(sobel)),
                "file_kb": os.path.getsize(row["filepath"]) / 1024,
                "resolution": f"{w}x{h}",
            }
        )
    return pd.DataFrame(stats)


eda_map = {}
stat_map = {}
for name, root in EDA_DATASETS.items():
    eda_map[name] = build_eda_df(root)
    stat_map[name] = build_stat_df(eda_map[name], sample_size=500)
    print(f"{name}: total images={len(eda_map[name])}, sampled_for_stats={len(stat_map[name])}, root={root}")

# Visual A: split distribution (concurrent)
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=False)
for ax, name in zip(axes, EDA_DATASETS.keys()):
    sns.countplot(data=eda_map[name], x="split", order=["train", "val", "test"], palette="Set2", ax=ax)
    ax.set_title(f"{name}: images per split")
    ax.set_xlabel("Split")
    ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

# Visual B: class distribution (concurrent)
fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=False)
for ax, name in zip(axes, EDA_DATASETS.keys()):
    order_labels = sorted(eda_map[name]["label"].unique())
    sns.countplot(data=eda_map[name], x="label", order=order_labels, palette="Set3", ax=ax)
    ax.set_title(f"{name}: images per class")
    ax.set_xlabel("Label")
    ax.set_ylabel("Count")
    ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()

# Visual C: width-height density (concurrent)
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=False, sharey=False)
for ax, name in zip(axes, EDA_DATASETS.keys()):
    sdf = stat_map[name]
    hb = ax.hexbin(sdf["width"], sdf["height"], gridsize=25, cmap="viridis")
    ax.set_title(f"{name}: width vs height density")
    ax.set_xlabel("Width (px)")
    ax.set_ylabel("Height (px)")
    cb = fig.colorbar(hb, ax=ax)
    cb.set_label("Count")
plt.tight_layout()
plt.show()

# Visual D: aspect ratio distribution (concurrent)
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=False)
for ax, name in zip(axes, EDA_DATASETS.keys()):
    sns.histplot(stat_map[name]["aspect"], bins=30, kde=True, color="mediumseagreen", ax=ax)
    ax.set_title(f"{name}: aspect ratio (W/H)")
    ax.set_xlabel("Aspect ratio")
    ax.set_ylabel("Frequency")
plt.tight_layout()
plt.show()

# Visual E: RGB mean distributions (concurrent)
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=False)
for ax, name in zip(axes, EDA_DATASETS.keys()):
    sdf = stat_map[name]
    sns.histplot(sdf["mean_r"], bins=25, color="red", alpha=0.45, ax=ax, label="R")
    sns.histplot(sdf["mean_g"], bins=25, color="green", alpha=0.45, ax=ax, label="G")
    sns.histplot(sdf["mean_b"], bins=25, color="blue", alpha=0.45, ax=ax, label="B")
    ax.set_title(f"{name}: mean RGB distributions")
    ax.set_xlabel("Channel mean")
    ax.set_ylabel("Frequency")
    ax.legend()
plt.tight_layout()
plt.show()

# Visual F: texture proxy comparison (brightness, contrast, edge)
texture_cols = ["brightness_v", "contrast_gray", "edge_strength"]
fig, axes = plt.subplots(len(texture_cols), 2, figsize=(14, 10), sharex=False)
for r, col in enumerate(texture_cols):
    for c, name in enumerate(EDA_DATASETS.keys()):
        ax = axes[r, c]
        sns.boxplot(data=stat_map[name], x="label", y=col, palette="Set2", ax=ax)
        ax.set_title(f"{name}: {col} by class")
        ax.set_xlabel("Label")
        ax.set_ylabel(col)
        ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()

# Visual G: top resolutions (concurrent)
fig, axes = plt.subplots(1, 2, figsize=(16, 4), sharey=False)
for ax, name in zip(axes, EDA_DATASETS.keys()):
    stat_map[name]["resolution"].value_counts().head(10).plot(kind="bar", ax=ax, color="teal")
    ax.set_title(f"{name}: top 10 resolutions")
    ax.set_xlabel("Resolution")
    ax.set_ylabel("Count")
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

eda_summary_rows = []
for name in EDA_DATASETS.keys():
    edf = eda_map[name]
    sdf = stat_map[name]
    eda_summary_rows.append(
        {
            "dataset": name,
            "total_images": int(len(edf)),
            "num_classes": int(edf["label"].nunique()),
            "sampled_stats_rows": int(len(sdf)),
            "mean_width": float(sdf["width"].mean()),
            "mean_height": float(sdf["height"].mean()),
            "mean_brightness": float(sdf["brightness_v"].mean()),
            "mean_edge_strength": float(sdf["edge_strength"].mean()),
        }
    )

eda_parity_summary_df = pd.DataFrame(eda_summary_rows).sort_values("dataset")
print("EDA parity summary:")
display(eda_parity_summary_df)

## Full dual-dataset transfer training and model comparison
Trains the full model set on each dataset separately (CNN + transfer models + ViT), saves each artifact, and builds a single comparison table/plots across all model-dataset pairs.

In [ ]:
# Full dual-dataset multi-model training pipeline
from pathlib import Path
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    Conv2D,
    MaxPooling2D,
    BatchNormalization,
    Flatten,
    Dense,
    Dropout,
    GlobalAveragePooling2D,
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import layers
from tensorflow.keras.applications import (
    EfficientNetB0,
    DenseNet121,
    VGG16,
    InceptionV3,
    MobileNetV2,
    ResNet50,
)

sns.set_style("darkgrid")

TRAIN_DATASETS = {
    "dataset1": Path("dataset1_70_20_10") if Path("dataset1_70_20_10").exists() else Path("dataset1"),
    "dataset2": Path("dataset2_70_20_10") if Path("dataset2_70_20_10").exists() else Path("dataset2"),
}

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS_PER_MODEL = 1  # increase to 5/10+ for stronger convergence
LEARNING_RATE = 1e-4
MODEL_DIR = Path("models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)


def make_generators(dataset_root: Path, img_size=IMG_SIZE, batch_size=BATCH_SIZE):
    train_dir = dataset_root / "train"
    val_dir = dataset_root / "val"
    test_dir = dataset_root / "test"

    augment = ImageDataGenerator(
        rescale=1.0 / 255,
        rotation_range=10,
        width_shift_range=0.05,
        height_shift_range=0.05,
        zoom_range=0.10,
        horizontal_flip=True,
        fill_mode="nearest",
    )
    plain = ImageDataGenerator(rescale=1.0 / 255)

    train_ds = augment.flow_from_directory(
        train_dir,
        target_size=img_size,
        batch_size=batch_size,
        class_mode="categorical",
        shuffle=True,
    )
    val_ds = plain.flow_from_directory(
        val_dir,
        target_size=img_size,
        batch_size=batch_size,
        class_mode="categorical",
        shuffle=False,
    )
    test_ds = plain.flow_from_directory(
        test_dir,
        target_size=img_size,
        batch_size=batch_size,
        class_mode="categorical",
        shuffle=False,
    )
    return train_ds, val_ds, test_ds


def build_cnn(input_shape, num_classes):
    model = Sequential(name="CNN_baseline")
    model.add(Conv2D(32, (3, 3), activation="relu", padding="same", input_shape=input_shape))
    model.add(MaxPooling2D((2, 2)))
    model.add(BatchNormalization())

    model.add(Conv2D(64, (3, 3), activation="relu", padding="same"))
    model.add(MaxPooling2D((2, 2)))
    model.add(BatchNormalization())

    model.add(Conv2D(128, (3, 3), activation="relu", padding="same"))
    model.add(MaxPooling2D((2, 2)))
    model.add(BatchNormalization())

    model.add(Flatten())
    model.add(Dense(256, activation="relu"))
    model.add(Dropout(0.4))
    model.add(Dense(num_classes, activation="softmax"))
    return model


def build_transfer(base_cls, input_shape, num_classes, model_name):
    base = base_cls(weights="imagenet", include_top=False, input_shape=input_shape)
    base.trainable = False
    x = base.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(256, activation="relu")(x)
    x = Dropout(0.3)(x)
    out = Dense(num_classes, activation="softmax")(x)
    return Model(inputs=base.input, outputs=out, name=model_name)


def build_vit(input_shape, num_classes):
    patch_size = 16
    projection_dim = 64
    transformer_layers = 6
    num_heads = 4
    dropout_rate = 0.1
    num_patches = (input_shape[0] // patch_size) * (input_shape[1] // patch_size)

    def vit_mlp(x, hidden_units, dropout):
        for units in hidden_units:
            x = layers.Dense(units, activation="gelu")(x)
            x = layers.Dropout(dropout)(x)
        return x

    inputs = layers.Input(shape=input_shape)
    x = layers.Conv2D(filters=projection_dim, kernel_size=patch_size, strides=patch_size, padding="valid")(inputs)
    x = layers.Reshape((num_patches, projection_dim))(x)

    positions = tf.range(start=0, limit=num_patches, delta=1)
    pos_embedding = layers.Embedding(input_dim=num_patches, output_dim=projection_dim)(positions)
    x = x + pos_embedding

    for _ in range(transformer_layers):
        x1 = layers.LayerNormalization(epsilon=1e-6)(x)
        attn_output = layers.MultiHeadAttention(num_heads=num_heads, key_dim=projection_dim, dropout=dropout_rate)(x1, x1)
        x2 = layers.Add()([attn_output, x])
        x3 = layers.LayerNormalization(epsilon=1e-6)(x2)
        x3 = vit_mlp(x3, hidden_units=[projection_dim * 2, projection_dim], dropout=dropout_rate)
        x = layers.Add()([x3, x2])

    x = layers.LayerNormalization(epsilon=1e-6)(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(dropout_rate)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    return Model(inputs=inputs, outputs=outputs, name="ViT_Keras")


def get_model_builders(input_shape, num_classes):
    return {
        "cnn_baseline": lambda: build_cnn(input_shape, num_classes),
        "efficientnet_b0": lambda: build_transfer(EfficientNetB0, input_shape, num_classes, "EfficientNetB0_frozen"),
        "densenet121": lambda: build_transfer(DenseNet121, input_shape, num_classes, "DenseNet121_frozen"),
        "vgg16": lambda: build_transfer(VGG16, input_shape, num_classes, "VGG16_frozen"),
        "inception_v3": lambda: build_transfer(InceptionV3, input_shape, num_classes, "InceptionV3_frozen"),
        "mobilenet_v2": lambda: build_transfer(MobileNetV2, input_shape, num_classes, "MobileNetV2_frozen"),
        "resnet50": lambda: build_transfer(ResNet50, input_shape, num_classes, "ResNet50_frozen"),
        "vit_keras": lambda: build_vit(input_shape, num_classes),
    }


dual_training_results = {}
comparison_rows = []

for dataset_name, dataset_root in TRAIN_DATASETS.items():
    print("\n" + "=" * 90)
    print(f"Training all models for {dataset_name} at {dataset_root}")
    print("=" * 90)

    train_ds, val_ds, test_ds = make_generators(dataset_root)
    input_shape = IMG_SIZE + (3,)
    num_classes = len(train_ds.class_indices)

    builders = get_model_builders(input_shape, num_classes)
    dual_training_results[dataset_name] = {
        "dataset_root": str(dataset_root),
        "class_indices": train_ds.class_indices,
        "models": {},
    }

    for model_key, builder in builders.items():
        print(f"\n[{dataset_name}] -> {model_key}")
        model = builder()
        model.compile(
            optimizer=Adam(learning_rate=LEARNING_RATE),
            loss="categorical_crossentropy",
            metrics=["accuracy"],
        )

        history = model.fit(
            train_ds,
            validation_data=val_ds,
            epochs=EPOCHS_PER_MODEL,
            verbose=1,
        )

        test_loss, test_acc = model.evaluate(test_ds, verbose=0)

        model_save_path = MODEL_DIR / f"{model_key}_{dataset_name}.h5"
        model.save(model_save_path)

        dual_training_results[dataset_name]["models"][model_key] = {
            "history": history.history,
            "test_acc": float(test_acc),
            "test_loss": float(test_loss),
            "model_path": str(model_save_path),
        }

        comparison_rows.append(
            {
                "dataset": dataset_name,
                "model": model_key,
                "test_accuracy": float(test_acc),
                "test_loss": float(test_loss),
                "classes": num_classes,
                "train_samples": int(train_ds.samples),
                "val_samples": int(val_ds.samples),
                "test_samples": int(test_ds.samples),
            }
        )

        print(f"Saved: {model_save_path}")
        print(f"test_acc={test_acc:.4f} | test_loss={test_loss:.4f}")

comparison_df = pd.DataFrame(comparison_rows)
comparison_df = comparison_df.sort_values(["model", "dataset"]).reset_index(drop=True)

print("\nCombined model comparison across dataset1 and dataset2:")
display(comparison_df)

pivot_acc = comparison_df.pivot(index="model", columns="dataset", values="test_accuracy")
print("\nPivoted accuracy table (rows=model, cols=dataset):")
display(pivot_acc)

# Visual 1: grouped bar of test accuracy by model and dataset
plt.figure(figsize=(12, 5))
sns.barplot(data=comparison_df, x="model", y="test_accuracy", hue="dataset")
plt.title("Test accuracy by model across datasets")
plt.xlabel("Model")
plt.ylabel("Accuracy")
plt.ylim(0, 1)
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

# Visual 2: grouped bar of test loss by model and dataset
plt.figure(figsize=(12, 5))
sns.barplot(data=comparison_df, x="model", y="test_loss", hue="dataset")
plt.title("Test loss by model across datasets")
plt.xlabel("Model")
plt.ylabel("Loss")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

# Visual 3: accuracy heatmap (model x dataset)
plt.figure(figsize=(8, 6))
sns.heatmap(pivot_acc, annot=True, fmt=".3f", cmap="YlGnBu", vmin=0.0, vmax=1.0)
plt.title("Accuracy heatmap: models vs datasets")
plt.xlabel("Dataset")
plt.ylabel("Model")
plt.tight_layout()
plt.show()

# Visual 4: concurrent learning curves for the same model across datasets
for model_key in comparison_df["model"].unique():
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for dataset_name in TRAIN_DATASETS.keys():
        hist = dual_training_results[dataset_name]["models"][model_key]["history"]
        ep = range(1, len(hist.get("accuracy", [])) + 1)

        axes[0].plot(ep, hist.get("accuracy", []), marker="o", label=f"{dataset_name} train")
        axes[0].plot(ep, hist.get("val_accuracy", []), marker="x", linestyle="--", label=f"{dataset_name} val")
        axes[1].plot(ep, hist.get("loss", []), marker="o", label=f"{dataset_name} train")
        axes[1].plot(ep, hist.get("val_loss", []), marker="x", linestyle="--", label=f"{dataset_name} val")

    axes[0].set_title(f"{model_key}: accuracy curves")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Accuracy")
    axes[0].legend(fontsize=8)

    axes[1].set_title(f"{model_key}: loss curves")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Loss")
    axes[1].legend(fontsize=8)

    plt.tight_layout()
    plt.show()

print("\nAll per-dataset model artifacts are saved in the models/ directory.")